# Análise e Preparação de Datasets para Reconhecimento de Expressões Faciais

## 1. Introdução ao Pré-processamento de Imagens

Para garantir que os modelos de aprendizado de máquina operem de forma eficiente e justa, é fundamental pré-processar os dados. Este notebook aborda o pipeline de pré-processamento para reconhecimento de emoções faciais, incluindo:

- Carregamento dos dados
- Redimensionamento para 100×100 pixels 
- Conversão para escala de cinza
- Normalização dos valores dos pixels
- Balanceamento pelo número da menor classe
- Separação em treino e teste (estratificada);

**FER-2013**: contém 35.887 imagens de 48×48 pixels em tons de cinza, com 7 emoções: 'angry', 'disgust', 'fear', 'happy', 'sad', 'surprise', 'neutral'.

**RAF-DB**: aproximadamente 5.000 imagens anotadas em condições naturais ("in the wild") com maior diversidade de expressões e características demográficas. Classes similares ao FER.

Observações:
- Coloca early stopping 3 ou 5 épocas
- Coloca Validação
- Primeiro esplita e depois subamostragem no treino
- ReduceLROnPlateau (mínimo global) antes do early stopping diminuí a learning rate Pacience=10
- Procurar artigos para sustentar escolhas

In [ ]:
import os #acessar e manipular diretórios e arquivos do sistema
import cv2 # OpenCV para leitura e processamento de imagens
import numpy as np #manipulação de arrays
from tqdm import tqdm #barra de progresso para visualizar carregamento de imagens
from collections import Counter #conta elementos para saber o número de imagens por classe
from sklearn.model_selection import train_test_split #separa os dados em treino e teste
from sklearn.utils import shuffle #embaralha os dados e rótulos na mesma ordem
import matplotlib.pyplot as plt #gráficos


## Carregamento e Pré-processamento das Imagens

Esta função percorre as pastas de um dataset, redimensiona todas as imagens para 100x100 pixels, converte para escala de cinza e normaliza os valores dos pixels entre 0 e 1.

In [ ]:
def load_images_from_directory(dataset_path, image_size=(100, 100)):
    X = []
    y = []

    # Lista de emoções (nome das subpastas)
    classes = sorted(os.listdir(dataset_path))

    for label in classes:
        class_dir = os.path.join(dataset_path, label)
        if not os.path.isdir(class_dir):
            continue

        for filename in tqdm(os.listdir(class_dir), desc=f'Carregando {label}'):
            filepath = os.path.join(class_dir, filename)

            # Lê imagem em escala de cinza
            image = cv2.imread(filepath, cv2.IMREAD_GRAYSCALE)
            if image is None:
                continue

            # Redimensiona para 100x100
            image = cv2.resize(image, image_size)

            # Normaliza os pixels (0 a 1)
            image = image / 255.0

            X.append(image)
            y.append(label)

    return np.array(X), np.array(y)


## Balanceamento por Subamostragem

Como o número de imagens por classe é desigual, utilizamos a técnica de subamostragem. Selecionamos a menor classe e reduzimos todas as demais para o mesmo número de exemplos, garantindo que o modelo não se incline para classes mais representadas.

In [ ]:
def balance_by_min_class(X, y):
    class_counts = Counter(y)
    min_class = min(class_counts.values())

    X_bal = []
    y_bal = []

    for emotion in class_counts.keys():
        idxs = np.where(y == emotion)[0]
        selected_idxs = np.random.choice(idxs, min_class, replace=False)

        X_bal.append(X[selected_idxs])
        y_bal.append(y[selected_idxs])

    X_bal = np.concatenate(X_bal, axis=0)
    y_bal = np.concatenate(y_bal, axis=0)

    # Embaralha os dados
    indices = np.arange(len(X_bal))
    np.random.shuffle(indices)

    return X_bal[indices], y_bal[indices]


## Ajuste de Formato para CNN

Expandimos a dimensão para incluir o canal (1), ficando com formato [N, 100, 100, 1].


In [ ]:
def prepare_for_cnn(X):
    return np.expand_dims(X, axis=-1)


## Divisão em Treino e Teste

Usamos o `train_test_split` com estratificação, para manter a proporção entre as classes nos dois conjuntos.


In [ ]:
def split_train_test(X, y, test_size=0.2, random_state=42):
    return train_test_split(X, y, test_size=test_size, stratify=y, random_state=random_state)


## Aplicação do Pipeline em um Dataset

Aqui aplicamos todas as etapas acima em um dataset. O caminho deve apontar para a pasta onde estão os diretórios com as emoções.


In [ ]:
# Caminho para o dataset 
dataset_path = 'dataset/FER2013'

# Etapa 1: Carregar as imagens
X, y = load_images_from_directory(dataset_path)

# Etapa 2: Balancear as classes
X_bal, y_bal = balance_by_min_class(X, y)

# Etapa 3: Preparar para CNN
X_bal = prepare_for_cnn(X_bal)

# Etapa 4: Separar treino e teste
X_train, X_test, y_train, y_test = split_train_test(X_bal, y_bal)

# Exibe resultados finais
print("Formato das imagens:", X_train.shape)
print("Classes:", np.unique(y_train))
print("Distribuição por classe:", Counter(y_train))


## Exemplo de imagens pré-processadas

Aqui visualizamos uma amostra do conjunto após redimensionamento, conversão e normalização.


In [ ]:
import matplotlib.pyplot as plt
import random

def show_sample_images(images, labels, class_names, samples_per_class=5):
    """
    Exibe uma grade com amostras aleatórias por classe, após o pré-processamento.

    Parâmetros:
    - images: array com as imagens pré-processadas (formato N x H x W x 1)
    - labels: rótulos (inteiros)
    - class_names: lista de nomes das classes (str)
    - samples_per_class: quantas imagens mostrar por classe
    """

    num_classes = len(class_names)
    plt.figure(figsize=(samples_per_class * 2, num_classes * 2))

    for class_idx, class_name in enumerate(class_names):
        # Indices de todas as imagens dessa classe
        idxs = [i for i, label in enumerate(labels) if label == class_idx]
        selected = random.sample(idxs, min(samples_per_class, len(idxs)))

        for i, idx in enumerate(selected):
            plt_idx = class_idx * samples_per_class + i + 1
            plt.subplot(num_classes, samples_per_class, plt_idx)
            image = images[idx].squeeze()  # remove o canal se for 1
            
            plt.imshow(image, cmap='gray')
            plt.title(class_name)
            plt.axis('off')

    plt.tight_layout()
    plt.show()

# Exemplo de uso
class_names = sorted(list(set(label_encoder.classes_)))  # nomes das classes
show_sample_images(images, labels, class_names)


## Salvando os Datasets Pré-processados

Após o pré-processamento completo — incluindo redimensionamento, normalização, conversão para escala de cinza, balanceamento e estratificação — é essencial salvar os conjuntos de treino e teste.

Isso garante reprodutibilidade, facilita o carregamento em outros notebooks e evita retrabalho em etapas pesadas como leitura e balanceamento das imagens.

A seguir, salvamos os arrays `X_train`, `y_train`, `X_test` e `y_test` como arquivos `.npz` (Numpy Zip), um formato leve e eficiente para armazenar múltiplos arrays.


In [ ]:
import numpy as np
import os

def save_preprocessed_data(X_train, y_train, X_test, y_test, output_dir='datasets_npz', prefix='dataset'):
    """
    Salva os conjuntos de dados pré-processados em arquivos .npz.

    Parâmetros:
    - X_train, y_train, X_test, y_test: arrays Numpy com os dados e rótulos
    - output_dir: pasta onde os arquivos serão salvos
    - prefix: prefixo usado nos nomes dos arquivos (ex: 'fer2013', 'rafdb', etc.)
    """
    os.makedirs(output_dir, exist_ok=True)

    # Caminho dos arquivos
    train_path = os.path.join(output_dir, f'{prefix}_train.npz')
    test_path = os.path.join(output_dir, f'{prefix}_test.npz')

    # Salva os arquivos
    np.savez_compressed(train_path, X=X_train, y=y_train)
    np.savez_compressed(test_path, X=X_test, y=y_test)

    print(f'[✔] Arquivos salvos com sucesso em: {output_dir}/')
    print(f' - {train_path}')
    print(f' - {test_path}')


In [ ]:
save_preprocessed_data(X_train, y_train, X_test, y_test, prefix='fer2013')
save_preprocessed_data(X_train, y_train, X_test, y_test, prefix='rafdb')
